# Practica 1 -  RA1: De datos sucios a dataset limpio 
**Alumno** : Gabriel Herrador
**Modulo** : M4 - Sistemas de Big Data

In [2]:
import pandas as pd
import numpy as np

# Lo primero q voy ha hacer sera cargar el database
df_raw = pd.read_csv('Dataset.csv')

# Lo segundo sera mostrar las columnas y las filas del dataset
print(f"Número de filas : {df_raw.shape[0]}")
print(f"Número de columnas: {df_raw.shape[1]}")

print("-----Primeras Filas-----")
print(df_raw.head())

print("\n-----Información de las columnas y Tipos de datos-----")
print(df_raw.info())

print("\n----- Valores ausentes y Unicos-----")
resumen = pd.DataFrame({
    'Tipos': df_raw.dtypes,
    'Valores Ausentes': df_raw.isna().sum(),
    'Unicos': df_raw.nunique()
})
display(resumen)

print(f"\nFilas duplicadas: {df_raw.duplicated().sum()}")

Número de filas : 150750
Número de columnas: 14
-----Primeras Filas-----
      product_id                product_name        brand  \
0  8006191567164         Pizza jamón y queso    Hacendado   
1  5010714108065             Pechuga de pavo      Navidul   
2  8010016840110          Helado de caramelo  Häagen-Dazs   
3  0010222575736  Pan de centeno sin corteza      Silueta   
4  4009934720080              Pasta integral    Hacendado   

                           category  country quantity_g energy_kcal_100g  \
0  Pizzas refrigeradas y congeladas   España        350            248.9   
1                         Embutidos   España        500            373.0   
2                           Helados  Bélgica        450            177.2   
3                               Pan   España        650            284.4   
4                             Pasta  Bélgica       1000            361.2   

  fat_100g sugars_100g proteins_100g salt_100g nutriscore_grade  \
0    13.29        4.57         10.39

,Tipos,Valores Ausentes,Unicos
product_id,str,96,149883
product_name,str,207,1808
brand,str,1994,421
category,str,717,113
country,str,531,47
quantity_g,str,366,272
energy_kcal_100g,str,673,7026
fat_100g,str,939,5706
sugars_100g,str,1043,6387
proteins_100g,str,833,3968



Filas duplicadas: 473


## Analisis detallado por cada columna 

### Columna: Product_id
Proposito: Indentificador unico del producto (Codigo de Barras/EAN)
Problemas detectados:
-  Prefijos "EAN-" innecesarios, guiones intercalados 
-  96 valores nulos 
Solucion: 
- Eliminar los prefijos "EAN-" e guiones intercalados 
- Comprobar si esos 96 valores nulos ya existen

In [14]:
df_raw['product_id'].head(15)
# Ver si hay códigos con letras o guiones
df_raw[df_raw['product_id'].astype(str).str.contains('[A-Za-z\-]', na=False)]['product_id'].head(29)

<>:3: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<>:3: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
C:\Users\GabrielHerradorÁlvar\AppData\Local\Temp\ipykernel_13028\579517792.py:3: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
  df_raw[df_raw['product_id'].astype(str).str.contains('[A-Za-z\-]', na=False)]['product_id'].head(29)


21       EAN-8002976215596
1740     EAN-0001237440768
1801        300045-4726805
2109        800319-3512956
2378        300007-8495118
2542        500067-0836317
4155     EAN-0001294853518
6108     EAN-3002951745884
6585        500029-8880884
7235     EAN-8004475282291
7976        400097-8489469
8247     EAN-5002216625119
8412     EAN-3001036060591
8586     EAN-3000898507565
9004        800187-2544564
9246     EAN-4002332638469
10925       400111-5884116
11694    EAN-8003301528114
11837    EAN-8000873641906
12083    EAN-4001313779924
12100    EAN-4002176079831
12494       300317-8941998
13393       500403-7203217
13819       800054-7616674
14281       500445-2475572
15979    EAN-8003492455206
16913    EAN-8004198671624
16976       400385-2611321
17478       300395-2390721
Name: product_id, dtype: str

### Columna: Product_name
Proposito: Nombre comercial del producto
Problemas detectados: 
- Hay 207 nulos  
-  casi 500 nombres con espacio sobrante
Solucion 
- Eliminar los espacios sobrantes y los q estan nulos cambiarlos a NAN

In [15]:
# Comprobaremos si hay valores nulos en la columna product_name
print(f"Nulos en product_name:{df_raw['product_name'].isna().sum()}")

# Ver si hay nombres con espacios raros al principio o al final
nombres_espacios = df_raw[df_raw['product_name'].str.strip() != df_raw['product_name']]
print(f"Nombres con espacios sobrantes: {len(nombres_espacios)}")

# Ver la muestra de la tabla las primeras 20 filas
print(df_raw.head(20))

Nulos en product_name:207
Nombres con espacios sobrantes: 487
       product_id                             product_name          brand  \
0   8006191567164                      Pizza jamón y queso      Hacendado   
1   5010714108065                          Pechuga de pavo        Navidul   
2   8010016840110                       Helado de caramelo    Häagen-Dazs   
3   0010222575736               Pan de centeno sin corteza        Silueta   
4   4009934720080                           Pasta integral      Hacendado   
5   8006455982573                             Leche entera      Carrefour   
6   5011367187995                          Helado de fresa         Magnum   
7   0010148295510                      Leche semidesnatada      Carrefour   
8   4004703824631                              Salsa César          Calvé   
9   5008810618038                               Espaguetis      Carrefour   
10  3002996646610                  Aceite de oliva intenso      Carrefour   
11  4003938849

### Columna: brand
Proposito: Marca o fabricante del producto 
Problemas:
- Hay 1994 nulos 
- Hay 2832 q estan con espacios sobrantes al inicio y al final 
Solucion: 
- Cambiar los nulos por "Desconocido"
- Eliminar los espacios sobrantes con la funcion strip 

In [16]:
# Comprobar si hay valores nulos en la columna brand
print(f"Nulos en brand:{df_raw['brand'].isna().sum()}")

# Comprobar si hay espacios sobrantes en la columna brand
marcas_espacios = df_raw[df_raw['brand'].str.strip() != df_raw['brand']]
print(f"Marcas con espacios sobrantes: {len(marcas_espacios)}")

# ver las marcas mas frecuentes
print("\n----- Marcas más frecuentes -----")
print(df_raw['brand'].value_counts(dropna=False).head(15))

Nulos en brand:1994
Marcas con espacios sobrantes: 2832

----- Marcas más frecuentes -----
brand
Hacendado                    24497
Carrefour                    23235
Nestlé                        3724
Kaiku                         2928
NaN                           1994
Puleva                        1639
Pascual                       1561
Central Lechera Asturiana     1536
Activia                       1456
Garofalo                      1403
Fanta                         1402
Panrico                       1397
Coca-Cola                     1393
Fontaneda                     1388
Gallo                         1376
Name: count, dtype: int64


### Columna: category
Proposito: Categoria de los productos
Problemas: 
- Hay 717 nulos 
- Hay 814 con espacios sobrantes en la parte del inicio y final
Solucion:
- Los valores nulos cambiarlos por desconocido 
- Eliminar los espacios sobrantes 

In [17]:
# Comprobar si hay valores nulos en la columna category
print(f"Nulos en category:{df_raw['category'].isna().sum()}")

# Comprobar si hay espacios sobrantes en la columna category
categorias_espacios = df_raw[df_raw['category'].str.strip() != df_raw['category']]
print(f"Categorías con espacios sobrantes: {len(categorias_espacios)}")

# ver las categorías mas frecuentes
print("\n----- Categorías más frecuentes -----")
print(df_raw['category'].value_counts(dropna=False).head(15))

Nulos en category:717
Categorías con espacios sobrantes: 814

----- Categorías más frecuentes -----
category
Yogures                             9535
Leche                               9495
Refrescos                           8142
Galletas                            8039
Patatas fritas y snacks             6917
Pan                                 6906
Pasta                               6843
Cereales                            6817
Chocolate                           6732
Quesos                              6730
Pizzas refrigeradas y congeladas    6702
Bebidas vegetales                   6212
Arroz                               6203
Salsas                              6141
Legumbres                           6118
Name: count, dtype: int64


### Columna: Country
Proposito:Pais de origen o comercialización del producto
Problema: 
- Inconsistencia de nombres o idiomas ('España', 'españa', 'spain', 'España/Spain,'ESP')
- 531 valores nulos
- 968 espacion sobrantes 
Solucion:
- limpiar espacios sobrantes
- Normalizar todos los nombres varianteso a una unica convencion, 
- Rellenar nulos con  'Desconocido' 

In [18]:
# Comprobaremos si hay valores nulos en la columna Country
print(f"Nulos en country:{df_raw['country'].isna().sum()}")

# Comprobar si hay espacios sobrantes en la columna country
paises_espacios = df_raw[df_raw['country'].str.strip() != df_raw['country']]
print(f"Paises con espacios sobrantes: {len(paises_espacios)}")

# ver los paises mas frecuentes
print("\n----- Paises más frecuentes -----")
print(df_raw['country'].value_counts(dropna=False).head(15))

Nulos en country:531
Paises con espacios sobrantes: 968

----- Paises más frecuentes -----
country
España            100045
Francia            11938
Italia             10525
Portugal            7371
Alemania            5928
Bélgica             3613
Reino Unido         2972
Países Bajos        2911
Irlanda             1526
Austria              749
NaN                  531
España / Spain       357
 españa              341
spain                328
ESP                  324
Name: count, dtype: int64


### Columna: quantity_g, energy_kcal_100, fat_100g, sugars_100g, proteins_100g, salt_100g
Proposito: Medidas de peso y composicion nutricional por cada 100g
Problemas encontrados: 
- Tipo de dato: Estan almacenados como textpo (str) en vez de numerico
- Valores ausentes tienen entre 300 y 1000 nulos cada una de las columnas
- Valores negativos incoherentes: Existen registros  con cantidades  o nutrientes negativos 
- Valores fuera del rango real: Pesos absurdos como 250kg y proporciones imposibles como  300g de azucar
Solucion:
- Convertir las columnas a tipo numerico
- Cambiar los valores incoherentes por 'NaN'  o por la media 

In [20]:
# Definimos la lista de columnas numéricas
cols_num = ['quantity_g', 'energy_kcal_100g', 'fat_100g', 'sugars_100g', 'proteins_100g', 'salt_100g']

# Ver el tipo de dato y cuántos nulos tiene cada una
for col in cols_num:
    print(f"--- Columna: {col} ---")
    print(f"Tipo de dato: {df_raw[col].dtype}")
    print(f"Nulos: {df_raw[col].isna().sum()}")
    # Convertimos temporalmente a numérico para ver min y max
    temp_s = pd.to_numeric(df_raw[col], errors='coerce')
    print(f"Valor mínimo: {temp_s.min()}")
    print(f"Valor máximo: {temp_s.max()}\n")

--- Columna: quantity_g ---
Tipo de dato: str
Nulos: 366
Valor mínimo: -500.0
Valor máximo: 250000.0

--- Columna: energy_kcal_100g ---
Tipo de dato: str
Nulos: 673
Valor mínimo: -50.0
Valor máximo: 9999.0

--- Columna: fat_100g ---
Tipo de dato: str
Nulos: 939
Valor mínimo: -5.0
Valor máximo: 250.0

--- Columna: sugars_100g ---
Tipo de dato: str
Nulos: 1043
Valor mínimo: -1.0
Valor máximo: 300.0

--- Columna: proteins_100g ---
Tipo de dato: str
Nulos: 833
Valor mínimo: -4.0
Valor máximo: 180.0

--- Columna: salt_100g ---
Tipo de dato: str
Nulos: 744
Valor mínimo: -0.5
Valor máximo: 90.0



### Columna: nutriscore_grade
Proposito: Clasificacion nutricional del producto (escala estandandarizada de la A a la E)
Problemas encontrados:
- Inconsistencia de formato: mezcla de minisculas y mayusculas(ej A y a)
- Presencia de espacios sobrantes en blanco
- Valores fuera de la escala normal(ej: 'f' '1') 
- Valores nulos 'NaN'
Solucion:
- Limpiar los espacios con ·'strip' y estandarizar mayusculas con el upper
- Cambiar los valores erroneos o no validos por 'NaN'

In [6]:
# 1.- Primero vamos a mirar cuantos nulos hay en la columna nutriscore_grade
print(f"Nulos en nutriscore_grade: {df_raw['nutriscore_grade'].isna().sum()}")

# 2.- vamos a mirar los diferentes valores que hay en la columna nutriscore_grade y  cuantas veces se repiten
print("\n----- Valores unicos de nutriscore_grade------")
print(df_raw['nutriscore_grade'].value_counts(dropna=False))

Nulos en nutriscore_grade: 1421

----- Valores unicos de nutriscore_grade------
nutriscore_grade
e          41404
d          32741
c          26524
b          26520
a          20658
NaN         1421
F            258
1            256
unknown      247
?            227
 e            68
 d            65
E             63
D             57
 c            52
B             46
C             43
 b            41
 a            33
A             26
Name: count, dtype: int64


### Columna: created_at , last_modified
Proposito: Marcas temporales de creación y ultima modificación del registro en el sistema
Problemas encontrados:
- Almacenadas como cadenas de texto ('str') en lugar del tipo de dato temporal correcto ('datetime)
- Presencia  de caracteres de formato tecnico ( 'T' y 'Z')
- **Algo positivo**: No hay nulos 
Solución:
- Convertir ambas columnas al formato fecha  y hora real de pandas mediante ' pd.to_datetime()

In [3]:
# Ver la informacion general de las columnas de  fecha
print("\n----- Columna: created_at -----")
print(f"Tipo de dato: {df_raw['created_at'].dtype} | Nulos: {df_raw['created_at'].isna().sum()}")
print(df_raw['created_at'].head(10))

print("\n----- Columna: last_modified -----")
print(f"Tipo de dato: {df_raw['last_modified'].dtype} | Nulos: {df_raw['last_modified'].isna().sum()}")
print(df_raw['last_modified'].head(10))


----- Columna: created_at -----
Tipo de dato: str | Nulos: 0
0    2016-07-17T00:05:09Z
1    2018-04-20T20:38:29Z
2    2016-11-04T11:24:49Z
3    2020-09-05T23:05:49Z
4    2018-02-16T11:37:18Z
5    2025-04-16T01:44:22Z
6    2018-07-15T10:23:59Z
7    2021-07-27T08:55:03Z
8    2018-03-10T19:38:05Z
9    2023-09-23T09:14:49Z
Name: created_at, dtype: str

----- Columna: last_modified -----
Tipo de dato: str | Nulos: 0
0    2018-01-06T00:49:06Z
1    2018-08-11T06:28:28Z
2    2016-12-19T22:13:04Z
3    2021-11-09T21:27:40Z
4    2018-03-18T17:15:28Z
5    2026-01-04T08:25:43Z
6    2018-09-10T09:57:27Z
7    2022-05-28T16:09:03Z
8    2019-08-05T06:35:15Z
9    2024-08-31T21:14:06Z
Name: last_modified, dtype: str


### Paso 1: Limpieza de textos ('product_name','brand','category')
* **Accion** Eliminacion de espacios en blanco al inicio  y final mediante 'str.strip()' y la sustitucion de los valores nulos (NaN) por la etiqueta '"Desconocido"' para no perder filas


In [3]:
# Creo un dataframe  de trabajo para la limpieza de datos 
df_clean = df_raw.copy()

# 1.- Limpiar la columna product_name
df_clean['product_name'] = df_clean['product_name'].str.strip()
df_clean['product_name'] = df_clean['product_name'].fillna('Desconocido')

# 2.- Limpiar la columna brand
df_clean['brand'] = df_clean['brand'].str.strip()
df_clean['brand'] = df_clean['brand'].fillna('Desconocido')

#3.- Limpiar la columna category
df_clean['category'] = df_clean['category'].str.strip()
df_clean['category'] = df_clean['category'].fillna('Desconocido')

print("¡Textos basicos limpiados correctamente!")
print(f"Nulos en product_name:{df_clean['product_name'].isna().sum()}")
print(f"Nulos en brand:{df_clean['brand'].isna().sum()}")
print(f"Nulos en category:{df_clean['category'].isna().sum()}")

¡Textos basicos limpiados correctamente!
Nulos en product_name:0
Nulos en brand:0
Nulos en category:0


### Paso 2: Normalizacion de 'country'
* **Accion** : Limpieza de espacios, estandarizacion de mayusculas/minusculas y unificando las variantes linguisticas ('españa', 'spain' 'ESP' , 'España/ Spain') a un unico valor oficial (' "España" '). Rellenar los nulos con ' "Desconocido" ' 

In [4]:
# 1.- Limpiamos los espacios y estandarizamos a formato titulo ( Primera letra en mayuscula)
df_clean['country'] = df_clean['country'].str.strip()

# 2.- Diccionario para unificar las variantes de España y otros paises comunes  si los hubiera 
mapeo_paises = {
    'españa': 'España',
    'spain': 'España',
    'ESP': 'España',
    'España / Spain': 'España',
    'france': 'Francia',
    'francia': 'Francia',
    'italy': 'Italia',
    'italia': 'Italia'
}

#  Aplicamos el remplazo de variantes 
df_clean['country'] = df_clean['country'].replace(mapeo_paises)

# 3.- Rellenamos los valores nulos con 'Desconocido'
df_clean['country'] = df_clean['country'].fillna('Desconocido')

# Verificamos los cambios realizados
print("---- Valores unicos de country limpios ----")
print(df_clean['country'].value_counts(dropna=False).head(10))

---- Valores unicos de country limpios ----
country
España          101395
Francia          11987
Italia           10582
Portugal          7371
Alemania          5928
Bélgica           3613
Reino Unido       2972
Países Bajos      2911
Irlanda           1526
Austria            749
Name: count, dtype: int64


### Paso 3: Limpieza y Validacion de las columnas numericas  y nutricionales 
* **Accion** : Conversión  de tipo texto  a numerico 'float' conversion de valores negativos e imposibles a NaN 

In [10]:
# Lista de columnas numéricas
cols_num = ['quantity_g', 'energy_kcal_100g', 'fat_100g', 'sugars_100g', 'proteins_100g', 'salt_100g']

for col in cols_num:
    # 1. Convertimos a numérico plano usando .to_numpy() para romper el enlace con PyArrow
    temp_numerico = pd.to_numeric(df_clean[col], errors='coerce').to_numpy()
    
    # 2. Reasignamos la columna convertida al DataFrame
    df_clean[col] = temp_numerico

# 3. Aplicamos los filtros de valores negativos de forma segura
for col in cols_num:
    df_clean.loc[df_clean[col] < 0, col] = float('nan')

# 4. Restricciones físicas lógicas por cada 100g de producto
nutrientes_100g = ['fat_100g', 'sugars_100g', 'proteins_100g', 'salt_100g']
for col in nutrientes_100g:
    df_clean.loc[df_clean[col] > 100, col] = float('nan')

# Calorías máximas lógicas en 100g (máximo 900 kcal)
df_clean.loc[df_clean['energy_kcal_100g'] > 900, 'energy_kcal_100g'] = float('nan')

# Comprobación de mínimos y máximos tras la limpieza
print("--- Valores tras la limpieza numérica ---")
print(df_clean[cols_num].describe().T[['min', 'max']])

--- Valores tras la limpieza numérica ---
                  min        max
quantity_g        0.0  250000.00
energy_kcal_100g  0.0     891.50
fat_100g          0.0     100.00
sugars_100g       0.0      89.87
proteins_100g     0.0      43.87
salt_100g         0.0      90.00


### Paso 4: Limpieza de 'nutriscore_grade'
* **Accion** : Estandarizacion de mayusculas , eliminacion de espacios y filtrado de valores fuera de la escala oficial por 'NaN'

In [11]:
# 1.- Limpieza de espacios y pasar todo a mayusculas
df_clean['nutriscore_grade'] = df_clean['nutriscore_grade'].str.strip().str.upper()

nutriscore_validos = ['A', 'B', 'C', 'D', 'E']

# 2.- Todo lo q no este dentro de la lista de valores validos lo pasamos a NaN
df_clean.loc[~df_clean['nutriscore_grade'].isin(nutriscore_validos), 'nutriscore_grade'] = pd.NA

# Comprobacion del resultado limpio
print("\n----- Valores  limpios de nutriscore_grade -----")
print(df_clean['nutriscore_grade'].value_counts(dropna=False))


----- Valores  limpios de nutriscore_grade -----
nutriscore_grade
E      41535
D      32863
C      26619
B      26607
A      20717
NaN     2409
Name: count, dtype: int64


### Paso 5: Conversion de fechas (created_at / last_modified)
* **Acción**: Transformacion de las cadenas de texto con formato ISO a objetos de tipo fecha/hora ('datetime')

In [13]:
# Convertir created_at y last_modified a  tipo datetime
df_clean['created_at'] = pd.to_datetime(df_clean['created_at'], errors='coerce')
df_clean['last_modified'] = pd.to_datetime(df_clean['last_modified'], errors='coerce')

print("----- Tipos de datos tras la conversión -----")
print(f"Tipo created_at: {df_clean['created_at'].dtype}")
print(f"Tipo last_modified: {df_clean['last_modified'].dtype}")

df_clean[['created_at', 'last_modified']].head(10)

----- Tipos de datos tras la conversión -----
Tipo created_at: datetime64[us, UTC]
Tipo last_modified: datetime64[us, UTC]


,created_at,last_modified
0,2016-07-17 00:05:09+00:00,2018-01-06 00:49:06+00:00
1,2018-04-20 20:38:29+00:00,2018-08-11 06:28:28+00:00
2,2016-11-04 11:24:49+00:00,2016-12-19 22:13:04+00:00
3,2020-09-05 23:05:49+00:00,2021-11-09 21:27:40+00:00
4,2018-02-16 11:37:18+00:00,2018-03-18 17:15:28+00:00
5,2025-04-16 01:44:22+00:00,2026-01-04 08:25:43+00:00
6,2018-07-15 10:23:59+00:00,2018-09-10 09:57:27+00:00
7,2021-07-27 08:55:03+00:00,2022-05-28 16:09:03+00:00
8,2018-03-10 19:38:05+00:00,2019-08-05 06:35:15+00:00
9,2023-09-23 09:14:49+00:00,2024-08-31 21:14:06+00:00


### Paso 6: Exportacion del dataset limpio 
* **Accion** : Guardado del dataframe transformado ("df_clean") en un nuevo archivo csv optimizado

In [14]:
# Guardar el dataframe limpio en un nuevo archivo CSV
nombre_archivo_salida = 'Dataset_limpio.csv'
df_clean.to_csv(nombre_archivo_salida, index=False)

print(f"Dataset limpio exportado con exito como '{nombre_archivo_salida}'")
print(f"Dimensiones del dataset limpio: {df_clean.shape[0]} filas y {df_clean.shape[1]} columnas")

Dataset limpio exportado con exito como 'Dataset_limpio.csv'
Dimensiones del dataset limpio: 150750 filas y 14 columnas
